# COMP0258 — ReMDM: Discrete Diffusion Planning in Craftax

> **Self-contained Colab notebook.** Loads pre-trained checkpoints from a public
> HuggingFace repo and demonstrates live inference, agent visualisation and the
> ReMDM denoising loop. No training is performed inside the notebook.

This notebook accompanies the paper *Discrete Diffusion as Non-Myopic Planners
for Procedurally Generated Environments*. It demonstrates a Remasking Discrete
Diffusion Model (**ReMDM**) acting as a planner in Craftax, and reproduces the
empirical core of the paper's Craftax chapter: **DAgger imitates the
PPO-RNN-1B expert with a measurable gap, and Advantage-Weighted Behavioural
Cloning (AWBC) RL fine-tuning fails to close it across a 25-ablation study.**

**How to use this notebook**

1. Open in Google Colab (preferably with a GPU runtime).
2. Run all cells top-to-bottom.
3. To test on **unseen inputs**, edit the constants in the *Configuration*
   cell (`SEED`, `ENV_NAME`, `EVAL_STEPS`, `EVAL_NUM_ENVS`, `DIFFUSION_STEPS_EVAL`)
   and re-run from there. Every Craftax seed produces a procedurally generated
   world the agent has never seen.

**Submission compliance**

- Cell 1 downloads everything from a single public HuggingFace repo
  (`HF_REPO_ID`) — no authentication required.
- The pre-trained checkpoint is loaded; no training happens here.
- Live inference (Cells 5–7) demonstrates that reported numbers reproduce.
- Pre-computed ablation figures (Cells 9–10) demonstrate the research finding.


In [1]:
# =============================================================================
# CONFIGURATION — marker can edit any of these and re-run from here
# =============================================================================

# Public HuggingFace repo containing src/, Craftax_Baselines/, configs/,
# checkpoints/ and pre-computed ablation outputs.
HF_REPO_ID = "MathisW78/remdm-craftax"
LOCAL_DIR = "remdm-craftax"

# --- Reproducibility & evaluation knobs ---
SEED = 42

# "Craftax-Classic-Symbolic-v1": 22 achievements, 17 actions  (DAgger checkpoint)
# "Craftax-Symbolic-v1":         65 achievements, 43 actions  (PPO expert only)
ENV_NAME = "Craftax-Classic-Symbolic-v1"

EVAL_STEPS = 2000          # Per-env step budget for live diffusion eval
EVAL_NUM_ENVS = 16         # Parallel environments (lower if Colab OOMs)
DIFFUSION_STEPS_EVAL = 10  # Reverse denoising steps at inference time


## 1. Project overview

### Problem

Plan action sequences in **Craftax**, a JAX-accelerated, procedurally generated
open-world survival game (a Crafter reimplementation extended with NetHack-like
mechanics). Two variants are used in the paper:

| Environment | Achievements | Actions | Notes |
|---|---|---|---|
| `Craftax-Classic-Symbolic-v1` | 22 | 17 | Crafter ported to JAX |
| `Craftax-Symbolic-v1`         | 65 | 43 | + NetHack mechanics, 9 floors |

### Approach — ReMDM as a non-myopic planner

A bidirectional **DenoisingTransformer** generates a `plan_horizon = 32` action
plan by iteratively denoising masked discrete tokens (MDLM / **ReMDM** —
Wang et al.) conditioned on the current symbolic observation. At inference,
MPC executes one action per plan and re-plans every 4 env steps with
**historical inpainting**: positions `0..hist_len-1` are locked to the actions
actually taken.

**Architecture (Craftax Classic).** `d_model=384`, `n_heads=8`, `n_layers=6`,
`d_ff=768`, 2-layer observation MLP encoder of width 768, sinusoidal time
embedding prefixing the action sequence, **cosine** noise schedule, dropout 0.1.
The **rescale** remasking strategy is used at sampling time with η=0.5 and
temperature=0.5.

### Pipeline (offline, before this notebook)

```
[1] PPO-RNN-1B expert            (Craftax_Baselines/ppo_rnn.py — 1B env steps)
[2] Offline Behavioural Cloning  (main.py --mode offline)
[3] Online DAgger fine-tuning    (main.py --mode online)
[4] AWBC RL fine-tuning ablation suite — 25 ablations
```

The diffusion planner is trained in two stages: first by **Offline BC** on
return-weighted PPO rollouts, then refined by **Online DAgger** with
exponentially-decaying expert mixing. RL fine-tuning uses
**Advantage-Weighted Behavioural Cloning (AWBC)**: trajectories weighted by
batch-normalised, clipped advantages.

### Three contributions (paper headline)

1. **Validation of generative planning.** ReMDM achieves an order of magnitude
   improvement over autoregressive baselines in MiniHack and the highest
   zero-shot OOD transfer (~8–10%) among all tested architectures.
2. **A double intractability.** A 25-ablation study across both environments
   shows that RL fine-tuning of masked discrete diffusion faces two compounding
   failures: an intractable log-likelihood forcing reliance on ELBO surrogates,
   *and* reward sparsity that renders these surrogates **informationally
   vacuous**.
3. **Characterisation of failure modes.** Partial-parameter methods
   catastrophically destroy the pretrained manifold; full-parameter methods
   preserve it but cannot improve upon it.

### Headline finding (Craftax)

DAgger fits the PPO-RNN-1B expert well enough to play Craftax meaningfully, but
asymptotes below the expert ceiling (≈12–13 vs ≈19 achievements on Classic),
and AWBC RL fine-tuning produces no meaningful improvement across **25
ablations** (best Δ over baseline RL = **+0.26**, well within seed variance).
**Group C** (capacity restrictions: head-only, attention-only, frozen backbone,
LoRA, …) **catastrophically collapses** with mean score **2.78** vs **10.82**
baseline; LoRA is the single worst at **−0.91**. This matches the parallel
finding from our MiniHack PyTorch codebase, indicating the obstruction is
framework-independent and points to the same **double intractability**.

The notebook demonstrates the items the marker needs to verify:

| Cell | Demonstrates |
|---|---|
| 5  | Live inference numbers reproduce reported results |
| 6  | The agent meaningfully interacts with Craftax (rewards, achievements) |
| 7  | The ReMDM iterative-unmasking sampler in action |
| 8  | DAgger vs PPO-RNN-1B expert (the imitation gap, live PPO eval) |
| 9–10 | Pre-computed ablation results — RL doesn't help |


In [5]:
# =============================================================================
# Install pinned dependencies, download the HF repo, set up sys.path
# =============================================================================

import importlib
import os
import subprocess
import sys


def _pip(*pkgs: str) -> None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *pkgs],
        stdout=subprocess.DEVNULL,
    )


# Core dependencies (versions match pyproject.toml).
_pip(
    "huggingface_hub>=1.9.1",
    "craftax>=1.5.0",
    "flax>=0.12.6",
    "optax>=0.2.8",
    "orbax-checkpoint>=0.5",
    "distrax>=0.1.7",
    "chex>=0.1.91",
    "polars>=1.39.3",
    "orjson>=3.11.8",
    "pyyaml>=6.0",
)

# JAX: keep Colab's preinstalled GPU build if present, otherwise install CPU.
try:
    import jax  # noqa: F401
    if jax.default_backend() == "gpu":
        pass
    else:
        raise RuntimeError("re-install needed")
except Exception:
    _pip("jax>=0.9.2")
    importlib.invalidate_caches()
    import jax  # noqa: F401

import craftax  # noqa: F401

backend = jax.default_backend()
device = jax.devices()[0]
print(f"JAX {jax.__version__} | backend={backend} | device={device}")
if backend != "gpu":
    print(
        "WARNING: JAX is running on CPU. Inference will be ~10x slower than GPU."
        "\n         In Colab, switch to a GPU runtime via Runtime -> Change runtime type."
    )

# ----------------------------------------------------------------------------
# Pull the project tree (code, configs, checkpoints, pre-computed results)
# from a single public HuggingFace repo. No authentication required.
# ----------------------------------------------------------------------------
from huggingface_hub import snapshot_download

snapshot_path = snapshot_download(repo_id=HF_REPO_ID, local_dir=LOCAL_DIR)
print(f"Snapshot downloaded to: {snapshot_path}")

# Both the parent (for `import src.planners...`) and the Craftax_Baselines/
# subdir (for `from wrappers import ...` inside Craftax_Baselines/ppo_rnn.py)
# must be on sys.path. main.py does the same thing.
for p in (snapshot_path, os.path.join(snapshot_path, "Craftax_Baselines")):
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(snapshot_path)
print(f"cwd: {os.getcwd()}")



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


JAX 0.9.2 | backend=cpu | device=TFRT_CPU_0
         In Colab, switch to a GPU runtime via Runtime -> Change runtime type.


Fetching 317 files: 100%|██████████| 317/317 [00:19<00:00, 16.43it/s]

Snapshot downloaded to: /Users/mathisweil/Documents/University/Data Science and Machine Learning MSc/Open-Endeness and General Intelligence/craftax-ReMDM-planner/remdm-craftax/remdm-craftax
cwd: /Users/mathisweil/Documents/University/Data Science and Machine Learning MSc/Open-Endeness and General Intelligence/craftax-ReMDM-planner/remdm-craftax/remdm-craftax


In [4]:
# =============================================================================
# Load the pre-trained DAgger diffusion checkpoint
# =============================================================================

import json

import jax
import jax.numpy as jnp
from craftax.craftax_env import make_craftax_env_from_name

from src.planners.model import build_model, load_checkpoint, make_apply_fns

# Checkpoint inventory bundled with the HF repo.
DIFFUSION_OFFLINE_CKPT = (
    "checkpoints/offline/Craftax-Classic-Symbolic-v1-OfflineDiffusion-BC-100M"
)
DIFFUSION_ONLINE_CKPT = (
    "checkpoints/online/Craftax-Classic-Symbolic-v1-OnlineDiffusion-DAgger-50M"
)
PPO_CKPT = {
    "Craftax-Classic-Symbolic-v1": (
        "checkpoints/ppo_agents/Craftax-Classic-Symbolic-v1-PPO_RNN-1000M"
    ),
    "Craftax-Symbolic-v1": (
        "checkpoints/ppo_agents/Craftax-Symbolic-v1-PPO_RNN-1000M"
    ),
}

# We pin to the DAgger (online) checkpoint — that is the headline result.
# The architecture hyperparameters are stored alongside the offline checkpoint
# in `resume_metadata.json` (online and offline share the same architecture).
with open(os.path.join(DIFFUSION_OFFLINE_CKPT, "resume_metadata.json")) as f:
    META = json.load(f)
ARCH_CFG = META["config_snapshot"]

# The diffusion checkpoints are trained on Classic Craftax. The denoiser's
# action head dimension is fixed at training time (num_actions = 17 for Classic),
# so we cannot transfer it to Full Craftax (43 actions). PPO covers both.
DIFFUSION_ENV_NAME = "Craftax-Classic-Symbolic-v1"

env_init = make_craftax_env_from_name(DIFFUSION_ENV_NAME, auto_reset=True)
env_params_init = env_init.default_params
NUM_ACTIONS = int(env_init.action_space(env_params_init).n)
OBS_DIM = int(env_init.observation_space(env_params_init).shape[0])
PLAN_HORIZON = int(ARCH_CFG["PLAN_HORIZON"])

print(f"DiffusionEnv : {DIFFUSION_ENV_NAME}")
print(f"  obs_dim    : {OBS_DIM}")
print(f"  num_actions: {NUM_ACTIONS}")
print(f"Architecture : d_model={ARCH_CFG['D_MODEL']} n_layers={ARCH_CFG['N_LAYERS']} "
      f"n_heads={ARCH_CFG['N_HEADS']} plan_horizon={PLAN_HORIZON}")

model = build_model(ARCH_CFG, NUM_ACTIONS)
apply_eval, _ = make_apply_fns(model)
diffusion_params = load_checkpoint(
    model,
    jax.random.PRNGKey(SEED),
    OBS_DIM,
    PLAN_HORIZON,
    DIFFUSION_ONLINE_CKPT,
)

n_params = sum(int(p.size) for p in jax.tree.leaves(diffusion_params))
print(f"Loaded DAgger checkpoint: {n_params / 1e6:.2f}M parameters")


SyntaxError: invalid syntax (1204906936.py, line 1)

In [ ]:
# =============================================================================
# CELL 5 (PRIORITY) — live inference on `EVAL_NUM_ENVS` parallel envs
# =============================================================================
#
# This calls the *exact* production inference code path used by
# `python main.py --mode inference`. The marker can change ENV_NAME / SEED /
# EVAL_STEPS in Cell 1 to test on entirely fresh procedurally generated worlds.
#
# Reported numbers (Craftax-Classic-Symbolic-v1, paper Tables 3 & 4):
#     Offline BC (15M params, 99M env steps)  ≈ 14–15 / 22 achievements
#     DAgger     (15M params, 99M env steps)  ≈ 12–13 / 22 achievements
#     PPO-RNN-1B expert ceiling               ≈ 19   / 22 achievements
# This notebook ships the DAgger checkpoint, so live inference should land in
# the 12–13 range. The gap to the PPO expert is the imitation gap discussed
# in §6 of the paper. EVAL_STEPS=2000 (default below) gives a noisier but
# fast estimate; bump to 10 000 for the headline number.
# -----------------------------------------------------------------------------

from src.planners.inference import run_inference

if "Classic" not in ENV_NAME:
    print(
        f"ENV_NAME={ENV_NAME!r}: no diffusion checkpoint exists for Full Craftax\n"
        f"(action vocabularies differ: 17 vs 43). Skipping diffusion eval — see\n"
        f"Cell 8 for the matching PPO expert evaluation on this environment."
    )
else:
    inference_cfg = {
        # Architecture (read from checkpoint metadata)
        **{k: v for k, v in ARCH_CFG.items() if k.isupper()},
        # Marker-controlled
        "ENV_NAME": ENV_NAME,
        "SEED": SEED,
        "EVAL_STEPS": EVAL_STEPS,
        "EVAL_NUM_ENVS": EVAL_NUM_ENVS,
        "DIFFUSION_STEPS_EVAL": DIFFUSION_STEPS_EVAL,
        # Always disable W&B inside the notebook
        "USE_WANDB": False,
        "CHECKPOINT_PATH": DIFFUSION_ONLINE_CKPT,
    }
    run_inference(inference_cfg)


In [ ]:
# =============================================================================
# CELL 6 (PRIORITY) — visualise the ReMDM planner acting in Craftax
# =============================================================================
#
# Re-runs the MPC + historical-inpainting loop on a small batch of envs and
# captures per-step rewards / achievements / actions for plotting. Each agent
# re-plans every step (DIFFUSION_STEPS_EVAL denoising steps per plan) and
# locks the prefix of executed actions via historical inpainting.
# -----------------------------------------------------------------------------

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from src.diffusion.sampling import sample_plan_inpainting

VIZ_NUM_ENVS = 4
VIZ_STEPS = 600

if "Classic" not in ENV_NAME:
    print(f"Skipping behaviour viz: no diffusion checkpoint for {ENV_NAME!r}.")
else:
    viz_env = make_craftax_env_from_name(DIFFUSION_ENV_NAME, auto_reset=True)
    viz_params = viz_env.default_params

    rng = jax.random.PRNGKey(SEED + 1)
    rng, env_rng = jax.random.split(rng)
    obs0, state0 = jax.vmap(viz_env.reset, in_axes=(0, None))(
        jax.random.split(env_rng, VIZ_NUM_ENVS), viz_params,
    )
    history0 = jnp.full((VIZ_NUM_ENVS, PLAN_HORIZON), NUM_ACTIONS, dtype=jnp.int32)
    hist_len0 = jnp.zeros(VIZ_NUM_ENVS, dtype=jnp.int32)
    env_indices = jnp.arange(VIZ_NUM_ENVS)

    @jax.jit
    def viz_step(carry, _):
        obs, state, rng, history, hist_len = carry
        rng, plan_rng, env_rng = jax.random.split(rng, 3)

        # Reset history when plan window is exhausted (matches inference.py).
        seq_full = hist_len >= PLAN_HORIZON
        hist_len = jnp.where(seq_full, 0, hist_len)
        history = jnp.where(seq_full[:, None], NUM_ACTIONS, history)

        plan = sample_plan_inpainting(
            apply_eval, diffusion_params, plan_rng, obs,
            history, hist_len, NUM_ACTIONS, PLAN_HORIZON,
            DIFFUSION_STEPS_EVAL,
            ARCH_CFG["TEMPERATURE"], ARCH_CFG["TOP_P"],
        )
        action = jnp.take_along_axis(plan, hist_len[:, None], axis=-1).squeeze(-1)
        history = history.at[env_indices, hist_len].set(action)
        hist_len = hist_len + 1

        obs_next, state_next, reward, done, _ = jax.vmap(
            viz_env.step, in_axes=(0, 0, 0, None),
        )(jax.random.split(env_rng, VIZ_NUM_ENVS), state, action, viz_params)

        hist_len = jnp.where(done, 0, hist_len)
        history = jnp.where(done[:, None], NUM_ACTIONS, history)
        return (
            (obs_next, state_next, rng, history, hist_len),
            (action, reward, done, state_next.achievements),
        )

    print(f"Rolling out {VIZ_NUM_ENVS} agents for {VIZ_STEPS} steps...")
    _, (acts, rews, dones, achs) = jax.lax.scan(
        viz_step, (obs0, state0, rng, history0, hist_len0), jnp.arange(VIZ_STEPS),
    )
    acts_np = np.array(acts)              # [T, E]
    rews_np = np.array(rews)              # [T, E]
    achs_np = np.array(achs)              # [T, E, num_ach]
    dones_np = np.array(dones)            # [T, E]

    # First-life cumulative reward + unlock count per agent
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    cum_reward = np.cumsum(rews_np, axis=0)
    unlock_count = achs_np.sum(axis=-1)
    for e in range(VIZ_NUM_ENVS):
        end = np.where(dones_np[:, e])[0]
        cutoff = int(end[0]) + 1 if len(end) > 0 else VIZ_STEPS
        axes[0].plot(np.arange(cutoff), cum_reward[:cutoff, e], label=f"agent {e}")
        axes[1].plot(np.arange(cutoff), unlock_count[:cutoff, e], label=f"agent {e}")
    axes[0].set_title("Cumulative reward (first life)")
    axes[0].set_xlabel("env step"); axes[0].set_ylabel("cumulative reward")
    axes[1].set_title("Achievements unlocked")
    axes[1].set_xlabel("env step"); axes[1].set_ylabel("# unique achievements")
    for ax in axes:
        ax.legend(loc="best", fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    # Action histogram (which actions does the planner actually use?)
    from craftax.craftax_classic.constants import Action as ClassicAction
    action_names = [a.name for a in ClassicAction]
    counts = np.bincount(acts_np.flatten(), minlength=NUM_ACTIONS)
    order = np.argsort(-counts)
    fig, ax = plt.subplots(figsize=(11, 3.5))
    ax.bar(range(NUM_ACTIONS), counts[order])
    ax.set_xticks(range(NUM_ACTIONS))
    ax.set_xticklabels([action_names[i] for i in order], rotation=70, ha="right", fontsize=8)
    ax.set_title(f"Action usage over {VIZ_STEPS * VIZ_NUM_ENVS} env steps")
    ax.set_ylabel("count"); ax.grid(alpha=0.3, axis="y")
    plt.tight_layout(); plt.show()


In [ ]:
# =============================================================================
# CELL 7 (PRIORITY) — visualise the ReMDM iterative unmasking process
# =============================================================================
#
# `sample_plan_inpainting` is JIT-compiled via lax.scan, so to capture the
# intermediate token sequences we re-implement its body as a Python loop. This
# is faithful to the production sampler — only the loop construct differs.
# Each row of the heatmap below is one denoising step: at step 0 the entire
# plan_horizon is masked; by the final step the plan has been fully resolved
# into action tokens, with ReMDM-style stochastic remasking interspersed.
# -----------------------------------------------------------------------------

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

if "Classic" not in ENV_NAME:
    print(f"Skipping denoising viz: no diffusion checkpoint for {ENV_NAME!r}.")
else:
    DENOISE_STEPS = 12  # show enough steps to see iterative unmasking
    MASK_ID = NUM_ACTIONS

    # Sample one observation from the env to condition on.
    viz_env = make_craftax_env_from_name(DIFFUSION_ENV_NAME, auto_reset=True)
    viz_params = viz_env.default_params
    one_rng = jax.random.PRNGKey(SEED + 2)
    obs1, _state1 = viz_env.reset(one_rng, viz_params)
    obs_b = obs1[None, :]                      # [1, obs_dim]

    seq = jnp.full((1, PLAN_HORIZON), MASK_ID, dtype=jnp.int32)
    history = jnp.full((1, PLAN_HORIZON), MASK_ID, dtype=jnp.int32)
    hist_len = jnp.zeros((1,), dtype=jnp.int32)
    rng = jax.random.PRNGKey(SEED + 3)
    temperature = float(ARCH_CFG["TEMPERATURE"])
    top_p = float(ARCH_CFG["TOP_P"])

    trace = [np.array(seq[0])]               # row 0 = fully masked

    # Mirror the body of src.diffusion.sampling.sample_plan_inpainting._step
    for step in range(1, DENOISE_STEPS + 1):
        rng, model_rng, sample_rng, remask_rng = jax.random.split(rng, 4)
        ratio = step / DENOISE_STEPS
        t_tensor = jnp.full((1,), 1.0 - ratio)
        logits = apply_eval(diffusion_params, obs_b, seq, t_tensor, model_rng) / max(temperature, 1e-8)

        # Nucleus filtering
        probs = jax.nn.softmax(logits, axis=-1)
        sorted_idx = jnp.argsort(-probs, axis=-1)
        sorted_p = jnp.take_along_axis(probs, sorted_idx, axis=-1)
        cutoff = jnp.cumsum(sorted_p, axis=-1) - sorted_p
        inv_idx = jnp.argsort(sorted_idx, axis=-1)
        nucleus_mask = jnp.take_along_axis(cutoff >= top_p, inv_idx, axis=-1)
        logits = jnp.where(nucleus_mask, -jnp.inf, logits)

        preds = jax.random.categorical(sample_rng, logits, axis=-1)
        conf = jnp.take_along_axis(
            jax.nn.softmax(logits, axis=-1), preds[..., None], axis=-1,
        ).squeeze(-1)
        num_unmask = max(1, int(PLAN_HORIZON * ratio))
        sorted_conf = jnp.sort(conf, axis=-1)[..., ::-1]
        thresh = sorted_conf[0, num_unmask - 1]
        seq_new = jnp.where(conf < thresh, MASK_ID, preds)

        # ReMDM-style remasking (matches sample_plan_inpainting)
        remask_prob = 0.15 * (1.0 - ratio)
        do_remask = (
            (jax.random.uniform(remask_rng, seq_new.shape) < remask_prob)
            & (seq_new != MASK_ID)
        )
        seq_new = jnp.where(do_remask, MASK_ID, seq_new)

        # Lock historical prefix (here: empty, so no-op)
        pos = jnp.broadcast_to(jnp.arange(PLAN_HORIZON)[None, :], (1, PLAN_HORIZON))
        seq_new = jnp.where(pos < hist_len[:, None], history, seq_new)

        seq = seq_new
        trace.append(np.array(seq[0]))

    trace = np.stack(trace)                  # [steps+1, plan_horizon]

    # Heatmap: rows = denoising step, columns = action position
    # Mask cells coloured grey, action cells coloured by token id (viridis).
    masked = trace == MASK_ID
    fig, ax = plt.subplots(figsize=(12, 5))
    cmap = plt.cm.viridis.copy()
    display = np.where(masked, np.nan, trace.astype(float))
    im = ax.imshow(
        display, aspect="auto", cmap=cmap, vmin=0, vmax=NUM_ACTIONS - 1,
        interpolation="nearest",
    )
    # Overlay grey for masked cells
    ax.imshow(
        np.where(masked, 1.0, np.nan), aspect="auto",
        cmap=ListedColormap(["#dddddd"]), vmin=0, vmax=1, interpolation="nearest",
    )
    ax.set_xlabel("plan position (action token)")
    ax.set_ylabel("denoising step")
    ax.set_title(
        f"ReMDM iterative unmasking ({DENOISE_STEPS} steps, plan_horizon={PLAN_HORIZON})"
        "\nGrey = MASK token, colour = sampled action ID"
    )
    cbar = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label("action ID")
    plt.tight_layout(); plt.show()

    n_masked = masked.sum(axis=1)
    print(f"Masked tokens per step: {list(n_masked)}")
    print(f"  step 0  (start): {int(n_masked[0])}/{PLAN_HORIZON} masked")
    print(f"  step {DENOISE_STEPS} (final): {int(n_masked[-1])}/{PLAN_HORIZON} masked")


In [ ]:
# =============================================================================
# CELL 8 — PPO-RNN-1B expert baseline (loaded live, evaluated on `ENV_NAME`)
# =============================================================================
#
# DAgger trains the diffusion planner to imitate this PPO-RNN expert (1B env
# steps of training in `Craftax_Baselines/`). We expect the DAgger return
# (Cell 5) to land below the PPO ceiling but in the same regime — there is a
# measurable imitation gap (DAgger ≈12–13 vs PPO ≈19 achievements on Classic).
# For Full Craftax, where no diffusion checkpoint exists (action vocabularies
# differ: 17 vs 43), this PPO baseline is the only evaluation that can run.
# -----------------------------------------------------------------------------

import os
import numpy as np
import jax
import jax.numpy as jnp
import orbax.checkpoint as ocp
from orbax.checkpoint import checkpoint_utils
from craftax.craftax_env import make_craftax_env_from_name

from src.planners.ppo import build_ppo_network, PPOAgent

PPO_NUM_ENVS = EVAL_NUM_ENVS
PPO_EVAL_STEPS = min(EVAL_STEPS, 1500)  # PPO eval is fast — bound it for snappy demo

ppo_path = os.path.abspath(PPO_CKPT[ENV_NAME])
ppo_env = make_craftax_env_from_name(ENV_NAME, auto_reset=True)
ppo_params_env = ppo_env.default_params
ppo_num_actions = int(ppo_env.action_space(ppo_params_env).n)
ppo_obs_dim = int(ppo_env.observation_space(ppo_params_env).shape[0])

# Build the PPO-RNN network and an abstract param pytree from a dummy init.
PPO_LAYER_SIZE = 512
ppo_net = build_ppo_network("ppo_rnn", ppo_num_actions, PPO_LAYER_SIZE,
                            {"LAYER_SIZE": PPO_LAYER_SIZE})
_dummy_x = (jnp.zeros((1, PPO_NUM_ENVS, ppo_obs_dim)),
            jnp.zeros((1, PPO_NUM_ENVS)))
_abstract_params = ppo_net.init(
    jax.random.PRNGKey(0),
    jnp.zeros((PPO_NUM_ENVS, PPO_LAYER_SIZE)),
    _dummy_x,
)

# Restore params only (the on-disk checkpoint also contains opt_state which we
# don't need for inference). `partial_restore=True` lets us read just `params`,
# and `construct_restore_args` is required on orbax >= 0.11 to provide sharding.
_restore_args = checkpoint_utils.construct_restore_args({"params": _abstract_params})
with ocp.CheckpointManager(ppo_path) as _mgr:
    _step = _mgr.latest_step()
    _restored = _mgr.restore(
        _step,
        args=ocp.args.PyTreeRestore(
            item={"params": _abstract_params},
            restore_args=_restore_args,
            partial_restore=True,
        ),
    )
print(f"Loaded PPO_RNN checkpoint from '{ppo_path}' (step {_step})")

ppo_agent = PPOAgent(
    network=ppo_net,
    params=_restored["params"],
    model_type="ppo_rnn",
    layer_size=PPO_LAYER_SIZE,
)

rng = jax.random.PRNGKey(SEED + 100)
rng, env_rng = jax.random.split(rng)
obs, state = jax.vmap(ppo_env.reset, in_axes=(0, None))(
    jax.random.split(env_rng, PPO_NUM_ENVS), ppo_params_env,
)
hidden0 = ppo_agent.init_hidden(PPO_NUM_ENVS)
done0 = jnp.zeros(PPO_NUM_ENVS, dtype=bool)


@jax.jit
def ppo_step(carry, _):
    obs, state, hidden, done, rng = carry
    rng, act_rng, env_rng = jax.random.split(rng, 3)
    action, hidden = ppo_agent.act(obs, done, hidden, act_rng, temperature=1.0)
    obs_next, state_next, reward, done_next, _ = jax.vmap(
        ppo_env.step, in_axes=(0, 0, 0, None),
    )(jax.random.split(env_rng, PPO_NUM_ENVS), state, action, ppo_params_env)
    return (obs_next, state_next, hidden, done_next, rng), (reward, done_next, state_next.achievements)


print(f"Running PPO expert: {PPO_NUM_ENVS} envs x {PPO_EVAL_STEPS} steps on {ENV_NAME}...")
_, (rewards, dones, achievements) = jax.lax.scan(
    ppo_step, (obs, state, hidden0, done0, rng), jnp.arange(PPO_EVAL_STEPS),
)

rewards_np = np.array(rewards)
dones_np = np.array(dones)
ach_np = np.array(achievements)

# First-life evaluation (matches src/planners/inference.py convention)
ep_returns = np.zeros(PPO_NUM_ENVS)
ep_unlocks = np.zeros(PPO_NUM_ENVS, dtype=int)
for i in range(PPO_NUM_ENVS):
    deaths = np.where(dones_np[:, i])[0]
    end = int(deaths[0]) if len(deaths) > 0 else PPO_EVAL_STEPS - 1
    ep_returns[i] = rewards_np[: end + 1, i].sum()
    ep_unlocks[i] = int(ach_np[: end + 1, i].max(axis=0).sum())

print()
print(f"PPO expert mean return : {ep_returns.mean():.2f}  (best={ep_returns.max():.2f})")
print(f"PPO expert mean unlocks: {ep_unlocks.mean():.2f} achievements")
if "Classic" in ENV_NAME:
    print()
    print(
        "Compare with the DAgger diffusion result printed in Cell 5. The paper\n"
        "reports (EVAL_STEPS=10000, 32 envs):\n"
        "  Offline BC ≈ 14–15 / 22 achievements\n"
        "  DAgger     ≈ 12–13 / 22 achievements\n"
        "  PPO-RNN-1B ≈ 19    / 22 achievements\n"
        "The gap between DAgger and PPO is the imitation gap — and the 25\n"
        "AWBC RL fine-tuning ablations in Cells 9–11 fail to close it."
    )


## 2. RL fine-tuning ablation suite (pre-computed)

Once DAgger had plateaued, we ran a **25-ablation suite** of
**Advantage-Weighted Behavioural Cloning (AWBC)** RL fine-tuning interventions
trying to push past it. Every figure and table below was generated offline
by `experiments/rl_finetuning/run_ablations.py` and shipped in the HF repo at
`experiments/rl_finetuning/outputs/craftax_classic_final_results/analysis/`.
Each ablation runs 500 iterations across 192 parallel envs and 3 seeds.

The four groups tested:

| Group | Hypothesis tested |
|---|---|
| **A** Regularisation | KL penalty, EWC, LLRD, LoRA, mixed replay, hard trust region |
| **B** Optimisation  | t-curriculum, entropy bonus, PCGrad, advantage clip, normalised adv, BC-on-wins, low-t |
| **C** Capacity      | head-only / FFN-only / attention-only / frozen backbone / top-k layer ablations |
| **D** Data quality  | reward filtering, action diversity, running stats, learned reward model |

### Headline result (Craftax Classic, Table 5 from the paper)

| Group | N | Mean | Δ vs Baseline RL | Best | Worst |
|---|---|---|---|---|---|
| Baseline RL          | – | 10.82 | –     | –     | –     |
| A (Regularisation)   | 6 |  8.85 | −1.97 | 10.99 | −0.91 |
| B (Training Signal)  | 7 | 10.01 | −0.82 | 10.92 |  5.07 |
| C (Architecture)     | 7 |  **2.78** | **−8.04** |  6.39 |  0.27 |
| D (Data Quality)     | 4 | 10.98 | +0.16 | 11.08 | 10.82 |

### Three findings (paper framing)

1. **The reward signal is informationally vacuous.** Groups A, B, and D all
   cluster within ±2 points of the 10.82 baseline. The single best ablation
   (`action_diversity`, **11.08**) improves by just **+0.26**. With clipped,
   batch-normalised advantages, AWBC degenerates to near-uniform BC on
   self-generated rollouts.
2. **Sharp partition into manifold preservation vs. catastrophic collapse.**
   Methods either remain above ~10 (manifold preserved) or fall below ~6.4
   (collapse). Group C's mean of **2.78** confirms severe structural collapse,
   but the 6.39 best-case shows that some partial functionality can survive
   before the manifold is destroyed.
3. **Partial-parameter methods universally collapse.** Group C drops **−8.04**
   from baseline; **LoRA** collapses to **−0.91** (worst overall; **−11.44**
   from the pretrained DAgger checkpoint). Freezing parameters does not
   preserve the pretrained representations — the denoising chain requires
   end-to-end full-parameter updates to stay coherent.

This is the empirical signature of the **double intractability** described in
the abstract: there is no closed-form `log π_θ` for masked discrete diffusion
(forcing an ELBO surrogate), *and* Craftax episode returns lack the variance
needed to make that surrogate informative. The two failure modes compound.


In [ ]:
# =============================================================================
# CELL 10 — Pre-computed ablation figures
# =============================================================================

import os
from IPython.display import Image, display, Markdown

ABLATION_DIR = "experiments/rl_finetuning/outputs/craftax_classic_final_results/analysis/figures"

KEY_FIGURES = [
    (
        "group_comparison.png",
        "**Group comparison** — final score by ablation group. Groups A, B "
        "and D cluster around the 10.82 baseline (Δ ∈ [−1.97, +0.16]). "
        "**Group C (capacity restrictions) collapses to mean 2.78** — the "
        "central catastrophic-collapse signature.",
    ),
    (
        "score_delta_over_baseline_rl.png",
        "**Δ-score over baseline RL** — every ablation, sorted. The best "
        "improvement is **+0.26** (`action_diversity`); **LoRA** is the "
        "single worst at **−11.44** vs the pretrained DAgger checkpoint. "
        "No ablation produces a meaningful improvement.",
    ),
    (
        "gradient_alignment.png",
        "**Gradient alignment** — cosine similarity between the BC loss and "
        "the AWBC RL loss gradients during fine-tuning. Frequently near zero "
        "or negative — consistent with an **informationally vacuous** reward "
        "signal that the AWBC weights cannot recover from.",
    ),
    (
        "gradient_conflict_map.png",
        "**Per-layer gradient conflict** — where in the network the BC and "
        "AWBC RL losses pull in opposite directions. Concentrated in the "
        "deeper backbone layers — the same layers Group C ablations restrict, "
        "explaining why partial-parameter methods collapse.",
    ),
]

for fname, caption in KEY_FIGURES:
    path = os.path.join(ABLATION_DIR, fname)
    if os.path.exists(path):
        display(Markdown(caption))
        display(Image(filename=path))
    else:
        display(Markdown(f"_Missing figure: `{fname}`_"))


In [ ]:
# =============================================================================
# CELL 11 — Ablation results tables (paper Tables 5 and 9)
# =============================================================================

import os
import polars as pl

TABLES_DIR = "experiments/rl_finetuning/outputs/craftax_classic_final_results/analysis/tables"

main_df = pl.read_csv(os.path.join(TABLES_DIR, "main_results.csv")).sort(
    "Final_Score", descending=True,
)
print("Main ablation results (sorted by Final_Score, ceiling = 22 achievements). Best = action_diversity at 11.08; baseline RL = 10.82; worst = LoRA at -0.91:")
print(main_df)

verdict_df = pl.read_csv(os.path.join(TABLES_DIR, "hypothesis_verdict.csv"))
print()
print("Hypothesis verdicts:")
print(verdict_df.select(["Ablation", "Group", "Result", "Conclusion"]))


## 3. Conclusions

1. **DAgger imitates, with a measurable gap.** The online DAgger checkpoint
   plateaus below the PPO-RNN-1B expert (≈12–13 vs ≈19 achievements on Classic
   Craftax). On full Craftax the gap is dramatic: Offline BC reaches ≈17–18
   achievements (≈11% of max reward 226), DAgger only ≈9–10 (≈5.3%), against a
   PPO ceiling of ≈10%. **Offline BC consistently outperforms DAgger** in
   Craftax — distributional shift from online aggregation hurts at scale,
   the opposite of MiniHack where BC and DAgger were comparable
   (69.3% vs 66.2% on the in-distribution suite). PPO-RNN reaches its ceiling
   in 15M–30M env steps; the diffusion planners need 80M–120M (**sample
   inefficiency**).
2. **AWBC RL fine-tuning is informationally vacuous.** Across **25 ablations**
   spanning regularisation, optimisation, capacity and data interventions, no
   method meaningfully improves on the DAgger checkpoint. The best Δ over
   baseline RL is **+0.26** (`action_diversity`), well within seed-to-seed
   variance. The clipped, batch-normalised AWBC weights degenerate to near-
   uniform — the surrogate carries no discriminative reward signal.
3. **Partial-parameter methods catastrophically collapse.** Group C (capacity
   restrictions) drops to mean **2.78** vs 10.82 baseline. **LoRA** collapses
   to **−0.91**, the worst overall result (**−11.44** from the pretrained
   checkpoint). Freezing the backbone does not protect the pretrained manifold
   — the denoising chain requires end-to-end full-parameter updates to stay
   coherent. EWC achieves the best Group A score (10.79) but its margin over
   baseline (+0.12) is negligible. Architectural interventions cannot
   compensate for the absence of a discriminative reward signal.
4. **Double intractability.** Combined with the parallel MiniHack PyTorch
   results, these experiments characterise the obstruction as fundamental:
   masked discrete diffusion has no closed-form `log π_θ` (forcing an ELBO
   surrogate), and reward sparsity in long-horizon procedurally generated
   environments renders that surrogate informationally vacuous. Current
   discrete diffusion planners are best understood as **imitation learning
   architectures**.

**Open problem.** A training objective for masked discrete diffusion that
permits stable RL fine-tuning without requiring `log π_θ` in closed form or
sufficient return variance. Promising directions: **Q-guided remasking**,
**SDE reformulations**, **categorical flow matching** (Campbell et al., 2024),
and **intrinsic motivation / hierarchical reward decomposition** to manufacture
the return variance the surrogate needs.

The full project documentation lives in the HF repo's `README.md`.
